# 27장 · 3D 메시 — 구멍 났나, 찢어졌나

이산위상 연구실 활용 편 · 글 안상선. 그대로 위에서부터 실행하면 돼요.

In [1]:
# 처음 한 번만: 필요한 패키지 설치 (구글 코랩이면 앞에 !를 붙여 그대로 실행)
# pip install numpy scipy matplotlib scikit-image networkx trimesh ripser

3D 물체는 수많은 삼각형(메시)으로 저장됩니다. **V − E + F**(오일러 수)와 **삼각형 하나에만 붙은 모서리**(가장자리)를 세면 구멍·찢어짐을 판정할 수 있어요. `trimesh`를 씁니다.

In [2]:
import numpy as np, trimesh

def report(name, m):
    V, E, F = len(m.vertices), len(m.edges_unique), len(m.faces)
    chi = V - E + F
    g = (2 - chi) / 2 if m.is_watertight else None
    print(f"{name:<14} V={V:>4} E={E:>4} F={F:>4}  V-E+F={chi:>3}  물샐틈없음={m.is_watertight}  손잡이 수={g}")

report('정육면체', trimesh.creation.box())
report('공(이십면체)', trimesh.creation.icosphere(subdivisions=2))
report('도넛', trimesh.creation.torus(major_radius=1.0, minor_radius=0.3))

정육면체           V=   8 E=  18 F=  12  V-E+F=  2  물샐틈없음=True  손잡이 수=0.0
공(이십면체)        V= 162 E= 480 F= 320  V-E+F=  2  물샐틈없음=True  손잡이 수=0.0
도넛             V=1024 E=3072 F=2048  V-E+F=  0  물샐틈없음=True  손잡이 수=1.0


공은 삼각형을 아무리 잘게 나눠도 V − E + F = 2, 도넛은 0입니다. 이제 공에서 삼각형 몇 개를 지워 「구멍 난 메시」를 만들고 가장자리 모서리를 찾습니다.

In [3]:
m = trimesh.creation.icosphere(subdivisions=2)
keep = np.ones(len(m.faces), bool); keep[:6] = False          # 삼각형 6개 지우기
hole = trimesh.Trimesh(m.vertices, m.faces[keep], process=True)
# 삼각형 하나에만 붙은 모서리 = 가장자리 (찢어지거나 뚫린 곳)
edges = np.sort(hole.edges, axis=1)
uniq, cnt = np.unique(edges, axis=0, return_counts=True)
boundary = uniq[cnt == 1]
report('구멍 난 공', hole)
print('가장자리 모서리 수:', len(boundary))

구멍 난 공         V= 162 E= 477 F= 314  V-E+F= -1  물샐틈없음=False  손잡이 수=None
가장자리 모서리 수: 12
